# Proyek Data Science: Prediksi Harga Rumah

**Tujuan:** membangun, mengevaluasi, membandingkan, dan menyiapkan model Machine Learning untuk memprediksi `SalePrice`.

**Dataset yang digunakan:** `train.csv`, `test.csv`, dan `data_description.txt` yang disediakan untuk latihan. Letakkan file CSV di folder yang sama dengan notebook ini.

## Workflow proyek
1. Memuat dan memahami data
2. Exploratory Data Analysis (EDA)
3. Memisahkan data latih dan validasi
4. Preprocessing menggunakan `Pipeline` dan `ColumnTransformer`
5. Modeling: Linear Regression dan Random Forest Regressor
6. Evaluation: MAE, RMSE, dan R²
7. Memilih model berdasarkan RMSE, lalu membandingkan metrik lainnya
8. Melatih model final, memprediksi `test.csv`, dan menyimpan artefak
9. Deployment demo menggunakan Streamlit

**Perbedaan dari latihan sebelumnya:** preprocessing dilakukan di dalam pipeline dan di-fit hanya pada data latih. Ini membantu mencegah *data leakage*.

## 0. Import library

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

## 1. Data Loading — memuat dataset

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Ukuran train:", train.shape)
print("Ukuran test :", test.shape)
display(train.head())

## 2. Data Understanding — memahami struktur data

In [ ]:
display(train.info())
display(train.describe(include="all").T.head(20))
print("Jumlah nilai kosong terbanyak:")
display(train.isnull().sum().sort_values(ascending=False).head(15))

## 3. EDA — analisis target dan pola data

In [ ]:
if "SalePrice" not in train.columns:
    raise ValueError("Kolom target 'SalePrice' tidak ditemukan di train.csv.")

plt.figure(figsize=(8, 4))
sns.histplot(train["SalePrice"], kde=True)
plt.title("Distribusi SalePrice")
plt.xlabel("SalePrice")
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(x=train["SalePrice"])
plt.title("Boxplot SalePrice")
plt.show()

# Hubungan beberapa fitur numerik dengan SalePrice
candidate_features = [c for c in ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF"] if c in train.columns]
for col in candidate_features:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(data=train, x=col, y="SalePrice", alpha=0.6)
    plt.title(f"{col} vs SalePrice")
    plt.show()

## 4. Menyiapkan fitur dan target

`SalePrice` adalah target yang akan diprediksi. Kolom `Id` hanya merupakan pengenal baris, sehingga tidak digunakan sebagai fitur. `test.csv` tidak digunakan untuk evaluasi karena biasanya tidak memiliki label `SalePrice`.

In [ ]:
X = train.drop(columns=["SalePrice", "Id"], errors="ignore")
y = train["SalePrice"].copy()

# Selaraskan kolom test dengan fitur train, dan buang Id jika ada
X_test_final = test.drop(columns=["Id"], errors="ignore").copy()
X_test_final = X_test_final.reindex(columns=X.columns)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Data latih:", X_train.shape)
print("Data validasi:", X_valid.shape)

## 5. Preprocessing Pipeline

- Fitur numerik: nilai kosong diisi median. Scaling digunakan untuk Linear Regression.
- Fitur kategori: nilai kosong diisi modus, lalu One-Hot Encoding.
- Semua preprocessing dipelajari dari data latih saja.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_preprocessor, numeric_features),
    ("cat", categorical_preprocessor, categorical_features)
])

print("Fitur numerik:", len(numeric_features))
print("Fitur kategori:", len(categorical_features))

## 6. Modeling — melatih dua model regresi

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=250,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=1
    )
}

fitted_models = {}
predictions = {}
results = []

for model_name, estimator in models.items():
    model_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_valid)

    fitted_models[model_name] = model_pipeline
    predictions[model_name] = y_pred
    results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_valid, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_valid, y_pred)),
        "R2": r2_score(y_valid, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df)

## 7. Evaluation — menentukan model yang paling sesuai

Untuk prediksi harga rumah, ketiga metrik bisa dipakai bersama:

- **MAE:** rata-rata besar kesalahan prediksi dalam satuan harga; mudah dipahami.
- **RMSE:** memberi penalti lebih besar pada kesalahan yang besar. Kita gunakan sebagai **metrik utama untuk memilih model** karena salah prediksi harga yang sangat jauh perlu diperhatikan.
- **R²:** menunjukkan seberapa besar variasi harga yang dapat dijelaskan model; semakin mendekati 1 umumnya semakin baik.

Tidak ada satu metrik yang selalu paling benar untuk semua kebutuhan. Karena itu, keputusan dibuat berdasarkan **RMSE terendah**, sambil tetap melaporkan MAE dan R². Nilai metrik aktual akan muncul setelah notebook dijalankan.

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_validation_model = fitted_models[best_model_name]

print("Model terpilih berdasarkan RMSE terendah:", best_model_name)
print(results_df.iloc[0])

### Visualisasi perbandingan hasil evaluasi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):
    sns.barplot(data=results_df, x="Model", y=metric, ax=ax)
    ax.set_title(f"Perbandingan {metric}")
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## 8. Dua referensi ilmiah untuk mendukung evaluasi

**Referensi 1 (2024)**  
*Advanced Machine Learning Techniques for Predictive Modeling of Property Prices*. *Information*, 15(6), 295. Artikel ini membahas pemodelan harga properti dan penggunaan MAE, RMSE, serta R² untuk mengevaluasi model.  
https://www.mdpi.com/2078-2489/15/6/295

**Referensi 2 (2025)**  
*Toward transparent and accurate housing price appraisal: Hedonic price models versus machine learning algorithms*. *Financial Innovation*. Artikel ini membandingkan pendekatan penilaian harga rumah dan menggunakan RMSE serta R² untuk menilai performa model.  
https://link.springer.com/article/10.1186/s40854-025-00874-w

Kedua artikel berada dalam rentang tahun yang diminta (2023–2026). Referensi ini mendukung penggunaan beberapa metrik, bukan menetapkan bahwa satu metrik selalu cocok untuk semua kasus.

## 9. Final Training — melatih model terpilih pada seluruh data berlabel

Setelah memilih jenis model dari data validasi, pipeline dibuat ulang dan dilatih pada seluruh `train.csv`. Hasil evaluasi yang dilaporkan tetap berasal dari validasi sebelumnya; pelatihan ulang ini dilakukan agar model akhir memanfaatkan seluruh data berlabel.

In [ ]:
if best_model_name == "Linear Regression":
    final_estimator = LinearRegression()
else:
    final_estimator = RandomForestRegressor(
        n_estimators=250,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=1
    )

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", final_estimator)
])
final_model.fit(X, y)

joblib.dump(final_model, "house_price_model.joblib")
print("Model tersimpan sebagai house_price_model.joblib")

## 10. Prediksi pada `test.csv` dan menyimpan hasil

`test.csv` biasanya tidak memiliki harga sebenarnya, jadi prediksi ini tidak bisa dipakai untuk menghitung MAE/RMSE/R². Hasilnya disimpan sebagai file submission bergaya kompetisi.

In [ ]:
test_predictions = final_model.predict(X_test_final)

submission = pd.DataFrame({
    "Id": test["Id"] if "Id" in test.columns else np.arange(1, len(test) + 1),
    "SalePrice": test_predictions
})
submission.to_csv("house_price_predictions.csv", index=False)

display(submission.head())
print("Hasil prediksi tersimpan sebagai house_price_predictions.csv")

## 11. Deployment — menyiapkan aplikasi Streamlit

Notebook ini membuat aplikasi demo yang memprediksi harga dari salah satu baris `test.csv`. Ini merupakan deployment sederhana untuk demonstrasi, bukan layanan produksi penuh.

Setelah menjalankan sel berikut:
1. Pastikan `house_price_model.joblib` dan `test.csv` ada di folder yang sama.
2. Install dependensi di terminal: `pip install streamlit pandas scikit-learn joblib`
3. Jalankan aplikasi: `streamlit run app.py`

In [ ]:
app_code = r"""
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(page_title="Prediksi Harga Rumah", page_icon="🏠")
st.title("Prediksi Harga Rumah")
st.write("Demo sederhana model Machine Learning untuk estimasi harga rumah.")

@st.cache_resource
def load_model():
    return joblib.load("house_price_model.joblib")

@st.cache_data
def load_test_data():
    return pd.read_csv("test.csv")

try:
    model = load_model()
    test_data = load_test_data()
    input_data = test_data.drop(columns=["Id"], errors="ignore")

    st.write("Pilih satu baris contoh dari test.csv untuk dibuat prediksi.")
    row_index = st.number_input(
        "Nomor baris (mulai dari 0)",
        min_value=0,
        max_value=max(0, len(input_data) - 1),
        value=0,
        step=1
    )
    sample = input_data.iloc[[int(row_index)]]
    st.subheader("Karakteristik rumah")
    st.dataframe(sample.T.rename(columns={sample.index[0]: "Nilai"}))

    if st.button("Prediksi harga"):
        prediction = model.predict(sample)[0]
        st.success(f"Estimasi SalePrice: ${prediction:,.0f}")
        st.caption("Ini adalah estimasi model, bukan penilaian resmi harga properti.")
except Exception as e:
    st.error("Aplikasi gagal dimuat. Pastikan app.py, house_price_model.joblib, dan test.csv berada dalam folder yang sama.")
    st.exception(e)
"""
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("File app.py berhasil dibuat.")

## 12. Kesimpulan

Setelah seluruh sel dijalankan, tuliskan kesimpulan berdasarkan angka yang benar-benar muncul:
- Model dengan RMSE terendah pada data validasi adalah ...
- MAE model tersebut adalah ...
- RMSE model tersebut adalah ...
- R² model tersebut adalah ...
- Model final disimpan sebagai `house_price_model.joblib`.
- Prediksi untuk `test.csv` disimpan sebagai `house_price_predictions.csv`.
- Aplikasi demo deployment tersedia pada `app.py`.

**Catatan:** jangan mengarang nilai evaluasi sebelum notebook dijalankan.